In [8]:
class PortfolioMetaEnv(gym.Env):
    """
    Môi trường huấn luyện Tổng Tư Lệnh phân bổ vốn đa tài sản (Chuẩn Gymnasium)
    """
    def __init__(self, n_assets=5, history_data=None):
        super(PortfolioMetaEnv, self).__init__()
        self.n_assets = n_assets
        self.data = history_data
        self.current_step = 0
        
        # Không gian hành động: Tỷ trọng phân bổ cho 5 đồng coin
        self.action_space = spaces.Box(low=0, high=1, shape=(self.n_assets,), dtype=np.float32)
        
        # Không gian quan sát: (5 coin x 3 tính năng = 15 chiều)
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(self.n_assets * 3,), dtype=np.float32)

    # 🛠️ FIX 1: Thêm seed và options, trả về (obs, info)
    def reset(self, seed=None, options=None):
        super().reset(seed=seed) # Kế thừa tính năng gieo hạt của Gymnasium
        self.current_step = np.random.randint(0, len(self.data) - 1000)
        return self._get_obs(), {} # Trả về Tuple (observation, info dict)

    def _get_obs(self):
        # Trải phẳng (flatten) ma trận 2D (5, 3) thành vector 1D (15,) để đưa cho SAC
        obs = self.data[self.current_step].flatten()
        return obs.astype(np.float32)

    def step(self, action):
        # Ép tỷ trọng về tổng = 1 (Softmax)
        weights = np.exp(action) / np.sum(np.exp(action))
        
        self.current_step += 1
        
        # Lấy lợi suất thực tế của nến tiếp theo (cột 0)
        next_returns = self.data[self.current_step, :, 0] 
        
        portfolio_return = np.sum(weights * next_returns)
        portfolio_volatility = np.std(next_returns)
        
        # Maximize lợi nhuận, Minimize rủi ro (Sharpe Ratio)
        reward = portfolio_return - (0.5 * portfolio_volatility) 
        
        # 🛠️ FIX 2: Tách 'done' thành 'terminated' và 'truncated'
        terminated = bool(self.current_step >= len(self.data) - 1)
        truncated = False 
        
        # Trả về đúng 5 tham số chuẩn Gymnasium
        return self._get_obs(), float(reward), terminated, truncated, {}

In [9]:
if __name__ == "__main__":
    print("📦 Đang chuẩn bị nguyên liệu cho Tổng Tư Lệnh...")
    
    # Ở đây mình tạo một Ma trận dữ liệu giả lập (Dummy Data) để bạn test thử trước.
    # Cấu trúc khối Rubik: (10000 thời điểm, 5 đồng coin, 3 tính năng)
    # 3 tính năng ví dụ: [Lợi suất (Return), Độ biến động (Vol), Tín hiệu (Signal)]
    
    num_steps = 10000
    n_assets = 5
    n_features = 3
    
    # Random dữ liệu giả lập cho 5 làn xe
    dummy_matrix_data = np.random.randn(num_steps, n_assets, n_features) 
    
    # 🚀 CHÂM LỬA !!!
    train_meta_commander(dummy_matrix_data)

📦 Đang chuẩn bị nguyên liệu cho Tổng Tư Lệnh...
🔥 Khởi động Lò rèn SAC Meta-Agent...
Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
🧠 Đang rèn luyện tư duy luân chuyển dòng tiền...
----------------------------------
| rollout/           |           |
|    ep_len_mean     | 4.44e+03  |
|    ep_rew_mean     | -1.87e+03 |
| time/              |           |
|    episodes        | 4         |
|    fps             | 18        |
|    time_elapsed    | 971       |
|    total_timesteps | 17765     |
| train/             |           |
|    actor_loss      | 2.19      |
|    critic_loss     | 0.112     |
|    ent_coef        | 0.00925   |
|    ent_coef_loss   | -1.94     |
|    learning_rate   | 0.0003    |
|    n_updates       | 17664     |
----------------------------------
----------------------------------
| rollout/           |           |
|    ep_len_mean     | 5.19e+03  |
|    ep_rew_mean     | -2.15e+03 |
| time/              |           |
| 